# Drift check (CUSUM)

The STNet fault head catches few drift faults: about 24% of drift events and 3% of drift cell-days at the 1-per-1,000 false-alarm budget. Longer LSTM training did not change that (notebook E). Most synthetic drift cell-days change the rain value by under 1 mm, so a check that reads one day at a time can't see them. This notebook scores an untrained check that adds up each cell's residual from its neighbours over days (a CUSUM, `qc/drift.py`) and compares it with the STNet fault head of the main run.

The rule is in `docs/qc-evaluation-protocol.md` ("Drift check (CUSUM)"), written before this run. Drift event recall must be at least 0.40 and 0.10 above STNet, drift PR-AUC 0.05 above, false alerts at most 2.0 per 1,000 clean cell-days, protected extremes flagged at most 2%. The parameters are fixed and are not tuned on these data. If the rule is not met, the wording is "no gain from this check".

Everything is on synthetic faults, validation years 2013 to 2017, one seed. A pass says nothing about real sensor drift.

## What it needs

- A GPU (T4 or P100) to score the STNet heads of the main run. The drift check itself is untrained and runs on the CPU. No training happens here.
- `stnet_s42_final.pt` from the main run, as a private dataset or a Kaggle model (framework PyTorch), not as a .zip. Put its mounted path from the Input panel in `RESUME_LINK` in the next cell.
- The RF25 dataset `nathashalewis/majorproj-rainfall`. Internet: On.

## How to run

1. New Notebook, File > Import Notebook, upload this file.
2. Accelerator: a single GPU. Add the two inputs above and set `RESUME_LINK`.
3. Save Version > Save & Run All (Commit). Preparing the benchmark takes about 13 minutes, the rest a few minutes more (an estimate from the earlier runs).
4. Download `stnet_results_drift.zip` from the Output tab. The decision is in `artifacts/qc/base/drift/drift_check_s42.md`.

In [ ]:
DATASET_LINK = '/kaggle/input/datasets/nathashalewis/majorproj-rainfall'
REPO_URL = 'https://github.com/NL0711/datamend.git'
BRANCH = 'main'
RUN_NAME = 'base'
EPOCHS, STEPS, LONG_STEPS = (4, 200, 30)
RESUME_LINK = ''
EXPECTED_BENCH_HASH = 'c37de7e3cf2be90e'
SEED = 42
print('run name:', RUN_NAME, '| seed:', SEED)

In [ ]:
import os, sys, subprocess, pathlib, shutil, json, time, importlib
WORK = pathlib.Path('/kaggle/working') if pathlib.Path('/kaggle/working').is_dir() else pathlib.Path.cwd()
def sh(cmd, cwd=None, show=True):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if show: print('$', cmd, chr(10) + out[-1200:])
    if r.returncode: raise RuntimeError('failed: ' + cmd[:120] + chr(10) + out[-1200:])
    return out
os.environ['GIT_TERMINAL_PROMPT'] = '0'
REPO = WORK / 'datamend'
try:
    if (REPO / '.git').is_dir():
        sh('git -C ' + str(REPO) + ' fetch ' + REPO_URL + ' ' + BRANCH + ' && git -C ' + str(REPO) + ' checkout -B ' + BRANCH + ' FETCH_HEAD')
    else:
        sh('git clone --branch ' + BRANCH + ' ' + REPO_URL + ' ' + str(REPO))
except RuntimeError as e:
    raise RuntimeError('Could not get the repository. Check that Internet is ON in the notebook settings.') from None
sh('git -C ' + str(REPO) + ' log -1 --oneline')
ML = REPO / 'ml-service'
os.chdir(ML)
sys.path.insert(0, str(ML))
for _m in [k for k in sys.modules if k == 'qc' or k.startswith('qc.')]:
    del sys.modules[_m]
importlib.invalidate_caches()
sh('pip install -q numpy pandas scipy scikit-learn xarray netCDF4 matplotlib pyyaml pyarrow kagglehub psutil nvidia-ml-py', show=False)
print('repo:', REPO, '| cwd:', os.getcwd())

In [ ]:
import torch, zipfile
from qc.kaggle_data import locate_rf25
from qc.config import load_config, year_range

cfg0 = load_config(ML / 'qc' / 'configs' / 'base.yaml', [])
need = sum(len(year_range(cfg0.years[k])) for k in ('train', 'val', 'test', 'stress') if cfg0.years.get(k))
def _download(slug):
    import kagglehub
    return kagglehub.dataset_download(slug)
RF25_DIR = locate_rf25(DATASET_LINK, expected_files=need, download=_download)
n_gpu = torch.cuda.device_count()
print('RF25:', RF25_DIR, '|', len(list(RF25_DIR.glob('RF25_ind*_rfp25.nc'))), 'files | GPUs:', n_gpu, [torch.cuda.get_device_name(i) for i in range(n_gpu)])
if n_gpu == 0: raise RuntimeError('No GPU. Scoring the STNet heads needs a CUDA GPU (T4 or P100). A TPU notebook cannot run it: change the accelerator in the notebook settings.')
os.environ['DATAMEND_RF25_DIR'] = str(RF25_DIR)
os.environ['DATAMEND_DATA_DIR'] = str(WORK / 'data')
OUT_ROOT = WORK / 'artifacts'
RUN = RUN_NAME
BASE = ['--profile', 'single-gpu', '--set', 'name=' + RUN]

if not RESUME_LINK.strip():
    raise RuntimeError('RESUME_LINK is empty. Attach the dataset or model that holds stnet_s42_final.pt and put its mounted path (Input panel) in RESUME_LINK in the first cell.')
root = pathlib.Path(RESUME_LINK.strip())
if not root.is_dir():
    raise RuntimeError('RESUME_LINK is not a folder here: ' + str(root) + '. Copy the path from the Input panel.')
stem = 'stnet_s' + str(SEED)
nm = stem + '_final'
ck = WORK / 'artifacts' / 'qc' / RUN / stem / 'checkpoints' / (nm + '.pt')
ck.parent.mkdir(parents=True, exist_ok=True)
if not ck.is_file():
    files = [f for f in sorted(root.rglob(nm + '.pt')) if f.is_file()]
    if files:
        shutil.copyfile(files[0], ck)
        print('weights file taken from', files[0])
    else:
        folders = [d for d in sorted(root.rglob(nm)) if d.is_dir() and (d / 'data.pkl').is_file()]
        if folders:
            with zipfile.ZipFile(ck, 'w', zipfile.ZIP_STORED) as z:
                for f in sorted(folders[0].rglob('*')):
                    if f.is_file():
                        z.write(f, pathlib.PurePosixPath(nm) / f.relative_to(folders[0]).as_posix())
            print('weights file rebuilt from the unzipped folder', folders[0])
if not ck.is_file():
    raise RuntimeError(nm + '.pt was not found under ' + str(root) + '. Upload the file as a private dataset or Kaggle model (not as a .zip).')
print('stage-1 checkpoint ready:', ck.stat().st_size // 1024, 'KB at', ck)

In [ ]:
def run_stream(step, *args):
    cmd = [sys.executable, '-u', '-m', 'qc', step, *args, *BASE]
    print('$', ' '.join(cmd))
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=ML)
    for line in p.stdout:
        print(line, end='')
    if p.wait():
        raise RuntimeError('step failed: ' + step)

run_stream('prepare')
run_stream('benchmark')

from qc.benchmark import bench_dir
cfg = load_config(ML / 'qc' / 'configs' / 'base.yaml', ['name=' + RUN])
bench_hash = json.loads((bench_dir(cfg) / 'manifest.json').read_text())['hash']
print('benchmark hash:', bench_hash)
if bench_hash != EXPECTED_BENCH_HASH:
    raise RuntimeError('benchmark hash ' + bench_hash + ' is not the expected ' + EXPECTED_BENCH_HASH + ': results from this notebook would not be comparable with the earlier ones. Stop and check the repository version.')

In [ ]:
RUN_ERRORS = []
try:
    run_stream('stnet', '--seeds', str(SEED), '--prefix', 'stnet', '--epochs', str(EPOCHS), '--steps', str(STEPS), '--long-steps', str(LONG_STEPS),
               '--set', 'compute.devices=[cuda:0]')
except Exception as ex:
    RUN_ERRORS.append('scoring the STNet heads failed: ' + str(ex)[:300])
    print('scoring the STNet heads failed:', str(ex)[:300])
if not RUN_ERRORS:
    try:
        run_stream('drift-check', '--base', 'stnet', '--seed', str(SEED), '--budget', '1.0')
    except Exception as ex:
        RUN_ERRORS.append('drift check failed: ' + str(ex)[:300])
        print('drift check failed:', str(ex)[:300])

In [ ]:
run_dir = OUT_ROOT / 'qc' / RUN
keep = []
for pat in ('**/*.md', '**/*.json', '**/*.png'):
    keep += [p for p in run_dir.glob(pat) if p.is_file() and p.stat().st_size < 50e6 and 'scores' not in p.parts and 'checkpoints' not in p.parts]
keep = list(dict.fromkeys(keep))
zname = WORK / 'stnet_results_drift.zip'
with zipfile.ZipFile(zname, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in keep:
        z.write(p, p.relative_to(WORK))
print(zname, '(' + format(zname.stat().st_size / 1e6, '.1f') + ' MB,', len(keep), 'files).')
report = run_dir / 'drift' / ('drift_check_s' + str(SEED) + '.md')
if report.is_file():
    print(report.read_text())
print('Download', zname.name, 'from the Output tab.')
if RUN_ERRORS:
    raise RuntimeError('; '.join(RUN_ERRORS))